# Experiment 08 — Improved EfficientNetB0 + Conservative SMOTE + XAI Evaluation

**Purpose:** test whether a safer SMOTE setup improves APTOS 2019 DR severity grading, then add paper-aligned evaluation and explainability outputs.

**Main design choices:**
1. RGB + LAB-CLAHE preprocessing is kept.
2. Data is split into train / validation / test before SMOTE.
3. SMOTE is applied only to the training split.
4. Validation and test sets remain untouched.
5. Oversampling is conservative: minority disease classes are raised toward the Moderate class count, not the No-DR majority count.
6. `k_neighbors=3` is used to avoid very broad interpolation in high-dimensional image space.
7. Class weights are disabled during SMOTE training to avoid double-compensating the minority classes.
8. Final evaluation is done once on the held-out test set.
9. Evaluation now includes multiclass ROC-AUC, saved confusion-matrix plots, Grad-CAM heatmaps, and optional quantitative XAI localization metrics.
10. Pointing Game and IoU are only computed when lesion masks are available. APTOS 2019 has image-level severity labels, not pixel-level lesion masks, so use an external lesion-mask dataset such as IDRiD for that part.


## 1. Environment Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
!pip install -q kaggle opencv-python pandas numpy tqdm scikit-learn imbalanced-learn matplotlib


In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/dr_severity_aptos")

RAW_DIR = PROJECT_DIR / "raw"
EXTRACTED_DIR = RAW_DIR / "aptos2019-blindness-detection"

# RGB + LAB-CLAHE cache. This is shared with exp07 because preprocessing is unchanged.
PROCESSED_DIR = PROJECT_DIR / "processed" / "aptos_224_rgb_lab_clahe"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# New experiment folder so the SMOTE run does not overwrite exp07.
EXPERIMENT_NAME = "exp08_smote_rgb_lab_clahe"
RESULTS_DIR = PROJECT_DIR / "results" / EXPERIMENT_NAME
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_FROZEN_PATH  = RESULTS_DIR / f"{EXPERIMENT_NAME}_frozen_best.keras"
MODEL_FINETUNE_PATH = RESULTS_DIR / f"{EXPERIMENT_NAME}_finetune_best.keras"
HISTORY_FROZEN_PATH  = RESULTS_DIR / f"{EXPERIMENT_NAME}_history_frozen.json"
HISTORY_FINETUNE_PATH = RESULTS_DIR / f"{EXPERIMENT_NAME}_history_finetune.json"
METRICS_PATH = RESULTS_DIR / f"{EXPERIMENT_NAME}_metrics.json"
REPORT_PATH  = RESULTS_DIR / f"{EXPERIMENT_NAME}_classification_report.csv"
CM_PATH      = RESULTS_DIR / f"{EXPERIMENT_NAME}_confusion_matrix.csv"
CONFIG_PATH  = RESULTS_DIR / f"{EXPERIMENT_NAME}_config.json"

TEST_METRICS_PATH = RESULTS_DIR / f"{EXPERIMENT_NAME}_test_metrics.json"
TEST_REPORT_PATH = RESULTS_DIR / f"{EXPERIMENT_NAME}_test_classification_report.csv"
TEST_CM_PATH = RESULTS_DIR / f"{EXPERIMENT_NAME}_test_confusion_matrix.csv"
TEST_CM_NORMALIZED_PATH = RESULTS_DIR / f"{EXPERIMENT_NAME}_test_confusion_matrix_normalized.csv"
TEST_CM_RAW_PNG_PATH = RESULTS_DIR / f"{EXPERIMENT_NAME}_test_confusion_matrix_raw.png"
TEST_CM_NORMALIZED_PNG_PATH = RESULTS_DIR / f"{EXPERIMENT_NAME}_test_confusion_matrix_normalized.png"
TEST_ROC_CURVE_PATH = RESULTS_DIR / f"{EXPERIMENT_NAME}_test_roc_curves.png"

GRADCAM_DIR = RESULTS_DIR / "gradcam_heatmaps"
GRADCAM_DIR.mkdir(parents=True, exist_ok=True)

# Optional: put lesion masks here if you want quantitative XAI evaluation.
# APTOS 2019 does not include lesion masks. IDRiD or another pixel-annotated DR dataset is needed.
LESION_MASK_DIR = PROJECT_DIR / "lesion_masks"
XAI_METRICS_PATH = RESULTS_DIR / f"{EXPERIMENT_NAME}_gradcam_xai_metrics.csv"
XAI_SUMMARY_PATH = RESULTS_DIR / f"{EXPERIMENT_NAME}_gradcam_xai_summary.json"


print("Results folder:", RESULTS_DIR)


## 2. Download & Extract Dataset

In [ ]:
import subprocess
import zipfile

RAW_DIR.mkdir(parents=True, exist_ok=True)
EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)

APTOS_ZIP = RAW_DIR / "aptos2019-blindness-detection.zip"

required_files = [
    EXTRACTED_DIR / "train.csv",
    EXTRACTED_DIR / "train_images"
]

if all(p.exists() for p in required_files):
    print("APTOS dataset already extracted:", EXTRACTED_DIR)
else:
    print("Dataset not found in:", EXTRACTED_DIR)
    print("Expected train.csv and train_images/.")
    print("If the Kaggle download fails, upload/extract the dataset manually to this folder.")

    # Kaggle CLI normally expects ~/.kaggle/kaggle.json.
    kaggle_json = Path.home() / ".kaggle" / "kaggle.json"

    if not kaggle_json.exists():
        print("Missing Kaggle credentials: ~/.kaggle/kaggle.json")
        print("Upload kaggle.json to Colab, then run this in a separate cell if needed:")
        print("!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/kaggle.json && chmod 600 ~/.kaggle/kaggle.json")
    else:
        if not APTOS_ZIP.exists():
            subprocess.run(
                [
                    "kaggle", "competitions", "download",
                    "-c", "aptos2019-blindness-detection",
                    "-p", str(RAW_DIR)
                ],
                check=True
            )

        if APTOS_ZIP.exists():
            print("Extracting dataset...")
            with zipfile.ZipFile(APTOS_ZIP, "r") as z:
                z.extractall(EXTRACTED_DIR)
            print("Extracted to:", EXTRACTED_DIR)

if not all(p.exists() for p in required_files):
    raise FileNotFoundError(
        f"APTOS dataset is still missing. Expected: {required_files}"
    )


## 3. Preprocessing — RGB + LAB-CLAHE

**FIX 1:** We now keep all three RGB channels instead of discarding red and blue.
CLAHE is applied to the L channel in LAB space, which enhances luminance contrast
without distorting hue — this preserves the clinically meaningful color differences
between lesion types (hemorrhages appear red, exudates appear yellow/white).
The green-channel-only approach was discarding two-thirds of the signal.

In [ ]:
import cv2
import numpy as np

IMG_SIZE = 224


def crop_black_border(img, threshold=10):
    """Remove black padding around fundus image."""
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    mask = gray > threshold
    if not mask.any():
        return img
    rows = np.where(mask.any(axis=1))[0]
    cols = np.where(mask.any(axis=0))[0]
    return img[rows[0]:rows[-1]+1, cols[0]:cols[-1]+1]


def resize_with_padding(img, target_size=224):
    """Aspect-ratio-preserving resize with zero padding."""
    h, w = img.shape[:2]
    scale = target_size / max(h, w)
    new_w, new_h = int(round(w * scale)), int(round(h * scale))
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    padded = np.zeros((target_size, target_size, 3), dtype=resized.dtype)
    y_off = (target_size - new_h) // 2
    x_off = (target_size - new_w) // 2
    padded[y_off:y_off+new_h, x_off:x_off+new_w] = resized
    return padded


def preprocess_fundus_rgb_lab_clahe(image_path, img_size=224):
    """
    FIX 1 — RGB + LAB-CLAHE preprocessing.

    Pipeline:
      1. Read as BGR, convert to RGB
      2. Crop black borders
      3. Resize with aspect-ratio padding
      4. Convert to LAB, apply CLAHE to L channel only, convert back to RGB
      5. Return uint8 RGB — all 3 channels intact

    Why LAB instead of green-channel:
      - L channel captures luminance contrast (same benefit as green channel CLAHE)
      - A/B channels preserve red-green and blue-yellow opponent colors
      - Hemorrhages, exudates, and neovascularization each have distinct color
        signatures that green-only preprocessing destroys entirely
    """
    img_bgr = cv2.imread(str(image_path))
    if img_bgr is None:
        raise ValueError(f"Could not read: {image_path}")

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_rgb = crop_black_border(img_rgb)
    img_rgb = resize_with_padding(img_rgb, img_size)

    # Apply CLAHE in LAB space to preserve color
    img_lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(img_lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    img_lab = cv2.merge([l, a, b])
    img_rgb = cv2.cvtColor(img_lab, cv2.COLOR_LAB2RGB)

    return img_rgb.astype(np.uint8)


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import Counter

# Optional: prevent OpenCV from overusing threads inside each worker
cv2.setNumThreads(1)

train_csv = EXTRACTED_DIR / "train.csv"
train_images_dir = EXTRACTED_DIR / "train_images"

df = pd.read_csv(train_csv)

print(df["diagnosis"].value_counts().sort_index())

X_PATH = PROCESSED_DIR / "X_rgb_lab_clahe_224.npy"
Y_PATH = PROCESSED_DIR / "y.npy"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 224
NUM_WORKERS = min(8, os.cpu_count() or 2)

print("Using workers:", NUM_WORKERS)


def process_one(row):
    image_path = train_images_dir / f"{row.id_code}.png"

    img = preprocess_fundus_rgb_lab_clahe(
        image_path,
        img_size=IMG_SIZE
    )

    label = int(row.diagnosis)

    return img, label


if X_PATH.exists() and Y_PATH.exists():
    print("Loading cached preprocessed data...")
    X = np.load(X_PATH)
    y = np.load(Y_PATH)

else:
    print("Preprocessing images in parallel...")

    rows = list(df.itertuples(index=False))

    X_list = [None] * len(rows)
    y_list = [None] * len(rows)

    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
        future_to_idx = {
            executor.submit(process_one, row): idx
            for idx, row in enumerate(rows)
        }

        for future in tqdm(as_completed(future_to_idx), total=len(rows)):
            idx = future_to_idx[future]

            try:
                img, label = future.result()
                X_list[idx] = img
                y_list[idx] = label

            except Exception as e:
                print(f"Error at index {idx}: {e}")
                raise e

    X = np.stack(X_list).astype(np.uint8)
    y = np.array(y_list, dtype=np.int64)

    np.save(X_PATH, X)
    np.save(Y_PATH, y)

    print("Cached.")

print("X:", X.shape, X.dtype, X.min(), X.max())
print("y:", y.shape, np.bincount(y))
print("Class distribution:", Counter(y))


## 4. Train / Validation / Test Split

The dataset is split before SMOTE. SMOTE is then applied only to the training data.
The validation set is used for checkpointing and early stopping. The test set is used only once for final evaluation.

This avoids data leakage: synthetic samples must never be generated from validation or test images.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter
import numpy as np

idx = np.arange(len(y))

# First split off a true test set: 15%.
train_val_idx, test_idx = train_test_split(
    idx,
    test_size=0.15,
    random_state=42,
    stratify=y
)

# Then split remaining 85% into train and validation.
# 0.1765 of 85% ≈ 15% of the total dataset.
train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=0.1765,
    random_state=42,
    stratify=y[train_val_idx]
)

X_train = X[train_idx].astype(np.float32)
X_val   = X[val_idx].astype(np.float32)
X_test  = X[test_idx].astype(np.float32)

y_train = y[train_idx].astype(np.int64)
y_val   = y[val_idx].astype(np.int64)
y_test  = y[test_idx].astype(np.int64)

# Keep these for documentation/comparison, but they are not used together with SMOTE.
class_weights_arr = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = {
    int(c): float(w)
    for c, w in zip(np.unique(y_train), class_weights_arr)
}

print("Train:", X_train.shape, Counter(y_train))
print("Val:", X_val.shape, Counter(y_val))
print("Test:", X_test.shape, Counter(y_test))
print("Original class weights, not used with SMOTE:", class_weights)


## 4b. Conservative SMOTE on Training Split Only

SMOTE is applied after preprocessing and after the train/validation/test split.
Only the training set is oversampled.

The APTOS 2019 distribution is strongly imbalanced, with No-DR much larger than Mild, Severe, and Proliferative DR. Instead of fully oversampling every class to the No-DR count, this experiment raises only the minority disease classes toward the Moderate class count. This avoids creating thousands of synthetic Severe/Proliferative images from very small classes.

Chosen SMOTE settings:

- `sampling_strategy`: oversample classes 1, 3, and 4 up to the current class-2 Moderate count.
- `k_neighbors=3`: conservative neighbor count for high-dimensional image vectors.
- No SMOTE on validation/test.
- No `class_weight` during SMOTE training, to avoid double weighting.


In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.utils import shuffle
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import gc

RANDOM_STATE = 42
NUM_CLASSES = 5

print("Before SMOTE:", Counter(y_train))
print("X_train range before SMOTE:", X_train.min(), X_train.max(), X_train.dtype)

# Work in [0, 1] for SMOTE interpolation, then convert back to [0, 255].
X_train_norm = np.clip(X_train / 255.0, 0.0, 1.0).astype(np.float32)

n_samples, height, width, channels = X_train_norm.shape
X_train_flat = X_train_norm.reshape(n_samples, -1)

class_counts = Counter(y_train)

# APTOS labels: 0 No DR, 1 Mild, 2 Moderate, 3 Severe, 4 Proliferative DR.
# Conservative target: raise minority disease classes to the Moderate count, not the No-DR majority count.
moderate_target = int(class_counts[2])
minority_disease_classes = [1, 3, 4]

sampling_strategy = {
    cls: moderate_target
    for cls in minority_disease_classes
    if class_counts[cls] < moderate_target
}

# Conservative k. SMOTE default is 5, but 3 is safer for high-dimensional image vectors.
if sampling_strategy:
    min_targeted_count = min(class_counts[cls] for cls in sampling_strategy)
    k_neighbors = min(3, min_targeted_count - 1)
else:
    k_neighbors = 0

print("SMOTE target count based on class 2 Moderate:", moderate_target)
print("SMOTE sampling_strategy:", sampling_strategy)
print("SMOTE k_neighbors:", k_neighbors)

if sampling_strategy and k_neighbors >= 1:
    smote = SMOTE(
        sampling_strategy=sampling_strategy,
        k_neighbors=k_neighbors,
        random_state=RANDOM_STATE
    )

    X_train_smote_flat, y_train_smote = smote.fit_resample(
        X_train_flat,
        y_train
    )

    X_train_smote = X_train_smote_flat.reshape(-1, height, width, channels)
    X_train_smote = np.clip(X_train_smote * 255.0, 0.0, 255.0).astype(np.float32)
    y_train_smote = y_train_smote.astype(np.int64)

    X_train_smote, y_train_smote = shuffle(
        X_train_smote,
        y_train_smote,
        random_state=RANDOM_STATE
    )
else:
    print("SMOTE was skipped because no class needed oversampling or k_neighbors was invalid.")
    X_train_smote = X_train.astype(np.float32)
    y_train_smote = y_train.astype(np.int64)

# Free large temporary arrays.
del X_train_norm, X_train_flat
gc.collect()

print("After SMOTE:", Counter(y_train_smote))
print("X_train_smote:", X_train_smote.shape, X_train_smote.dtype, X_train_smote.min(), X_train_smote.max())
print("y_train_smote:", y_train_smote.shape)

# These variables are used by the training cells.
X_train_fit = X_train_smote
y_train_fit = y_train_smote

# Do not use class_weight together with SMOTE in this experiment.
TRAIN_CLASS_WEIGHT = None

print("Training set used by model:", X_train_fit.shape, Counter(y_train_fit))
print("TRAIN_CLASS_WEIGHT:", TRAIN_CLASS_WEIGHT)


In [ ]:
# Visual sanity check for synthetic samples.
# Because the training set was shuffled after SMOTE, we identify likely synthetic samples by comparing count.
# For a quick check, show random samples from the oversampled classes in the SMOTE training set.

rng = np.random.default_rng(42)
classes_to_show = [1, 3, 4]

plt.figure(figsize=(12, 8))
plot_idx = 1

for cls in classes_to_show:
    cls_indices = np.where(y_train_fit == cls)[0]
    if len(cls_indices) == 0:
        continue

    chosen = rng.choice(cls_indices, size=min(4, len(cls_indices)), replace=False)

    for idx_img in chosen:
        plt.subplot(len(classes_to_show), 4, plot_idx)
        plt.imshow(np.clip(X_train_fit[idx_img] / 255.0, 0.0, 1.0))
        plt.title(f"Class {cls}")
        plt.axis("off")
        plot_idx += 1

plt.tight_layout()
plt.show()


## 5. Augmentation

**FIX 5:** `value_range=(0, 255)` is now correct because data is float32 in `[0, 255]`.
Previously data was in `[0, 1]` but `value_range=(0, 255)` was set, causing
brightness shifts of 38 (0.15 × 255) to be applied to values that only ranged 0–1.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

augmentation_layers = [
    layers.RandomRotation(factor=15/360),
    layers.RandomFlip(mode="horizontal_and_vertical"),
    layers.RandomTranslation(height_factor=0.05, width_factor=0.05),
    # value_range matches X_train/X_val range [0, 255]
    layers.RandomBrightness(factor=0.15, value_range=(0, 255)),
    layers.RandomContrast(factor=0.15),
]

# RandomErasing is not available in every TensorFlow/Keras version.
# Add it only when the local environment supports it.
if hasattr(layers, "RandomErasing"):
    augmentation_layers.append(
        layers.RandomErasing(
            factor=0.10,
            scale=(0.01, 0.04),
            fill_value=0.0,
            value_range=(0, 255)
        )
    )
else:
    print("RandomErasing is not available in this TensorFlow version; continuing without cutout.")

data_augmentation = tf.keras.Sequential(
    augmentation_layers,
    name="data_augmentation"
)


## 6. Validation QWK Callback

Quadratic weighted kappa is calculated at the end of each epoch with a callback.
This avoids using `.numpy()` inside a compiled Keras metric, which causes
`SymbolicTensor` errors in graph mode. The callback writes `val_qwk` into the
training logs, so `ModelCheckpoint` and `EarlyStopping` can still monitor it.

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import cohen_kappa_score

class ValidationQWK(tf.keras.callbacks.Callback):
    """
    Computes validation quadratic weighted kappa after each epoch.

    Do not implement QWK as a normal Keras metric using .numpy() inside
    update_state(). Keras often trains in graph mode, where tensors are symbolic.
    This callback predicts on the validation set after each epoch, so it can use
    sklearn safely and can still expose logs['val_qwk'] for checkpointing and
    early stopping.
    """
    def __init__(self, validation_data, batch_size=32):
        super().__init__()
        self.X_val, self.y_val = validation_data
        self.batch_size = batch_size

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}

        y_pred_probs = self.model.predict(
            self.X_val,
            batch_size=self.batch_size,
            verbose=0
        )
        y_pred = np.argmax(y_pred_probs, axis=1)

        y_true = self.y_val
        if len(y_true.shape) > 1:
            y_true = np.argmax(y_true, axis=1)

        qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
        logs["val_qwk"] = float(qwk)

        print(f" - val_qwk: {qwk:.4f}")

print("ValidationQWK callback ready.")


## 7. Model Architecture

**FIX 4:** Added a Dense(256) → BN → Dropout(0.5) → Dense(128) → Dropout(0.4) head.
The original single Dropout(0.3) → Dense(5) gives the model almost no capacity between
the backbone and output, causing it to overconfidently predict majority classes.

In [ ]:
from tensorflow.keras import layers, models

NUM_CLASSES = 5
INPUT_SHAPE = (224, 224, 3)


def build_model(trainable_from_layer=None):
    """
    Build EfficientNetB0 with an improved classification head.

    trainable_from_layer=None freezes the whole backbone.
    trainable_from_layer=int unfreezes backbone layers from that index onward.
    BatchNorm layers are always kept frozen for stability on this small dataset.
    """
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=INPUT_SHAPE
    )

    if trainable_from_layer is None:
        base_model.trainable = False
    else:
        base_model.trainable = True
        for layer in base_model.layers[:trainable_from_layer]:
            layer.trainable = False
        for layer in base_model.layers[trainable_from_layer:]:
            layer.trainable = True

    # Always keep BatchNorm frozen/in inference mode.
    for layer in base_model.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

    inputs = layers.Input(shape=INPUT_SHAPE, name="input_image")
    x = data_augmentation(inputs)

    # EfficientNetB0 in Keras includes its own preprocessing/rescaling internally
    # in recent TF/Keras versions. We keep images in [0,255] to match augmentation.
    x = base_model(x, training=False)

    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dense(256, activation="relu", name="dense_256")(x)
    x = layers.BatchNormalization(name="bn_head")(x)
    x = layers.Dropout(0.5, name="dropout_1")(x)
    x = layers.Dense(128, activation="relu", name="dense_128")(x)
    x = layers.Dropout(0.4, name="dropout_2")(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="output")(x)

    return models.Model(inputs, outputs, name="EfficientNetB0_Improved")


model = build_model(trainable_from_layer=None)
model.summary()


## 8. Phase 1 — Frozen Backbone Training

**FIX 3 & FIX 7:** Cosine decay schedule replaces `ReduceLROnPlateau(factor=0.1)`.
`factor=0.1` was dropping the LR from 1e-3 → 1e-5 in two steps, effectively
halting training. Cosine decay provides smooth, continuous LR reduction.
Max epochs increased to 60 (from 30) to allow full convergence.

In [ ]:
import json
import math

BATCH_SIZE = 32
FROZEN_EPOCHS = 60
steps_per_epoch_frozen = math.ceil(len(X_train_fit) / BATCH_SIZE)

# CosineDecay alpha is a fraction of the initial LR.
# 1e-3 * 1e-3 = 1e-6 final LR.
lr_schedule_frozen = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3,
    decay_steps=steps_per_epoch_frozen * FROZEN_EPOCHS,
    alpha=1e-3
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule_frozen),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks_frozen = [
    # Must run before ModelCheckpoint/EarlyStopping so logs['val_qwk'] exists.
    ValidationQWK(validation_data=(X_val, y_val), batch_size=BATCH_SIZE),

    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(MODEL_FROZEN_PATH),
        monitor="val_qwk",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_qwk",
        mode="max",
        patience=20,
        restore_best_weights=True,
        verbose=1
    )
]

print("Starting frozen backbone training...")

history_frozen = model.fit(
    X_train_fit,
    y_train_fit,
    validation_data=(X_val, y_val),
    epochs=FROZEN_EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=TRAIN_CLASS_WEIGHT,
    callbacks=callbacks_frozen
)

# Convert numpy scalars to normal Python values for JSON serialization.
history_frozen_dict = {
    k: [float(x) for x in v]
    for k, v in history_frozen.history.items()
}

with open(HISTORY_FROZEN_PATH, "w") as f:
    json.dump(history_frozen_dict, f, indent=4)

print("Phase 1 complete. Best model saved to:", MODEL_FROZEN_PATH)


## 9. Phase 2 — Fine-Tuning (Unfreeze from Layer 100)

**FIX 2:** Unfreeze from layer ~100 (last ~137 layers, covering the final 3 MBConv block
groups). The original code only unfroze the last 30 layers, leaving the mid-level
feature extractors — which need to adapt from ImageNet to fundus imagery — completely frozen.

**FIX 3:** Cosine decay with warm restarts for fine-tuning at very low initial LR (1e-4).
The original `Adam(lr=1e-5)` with `ReduceLROnPlateau(factor=0.1)` would drop to 1e-7
within a few plateau cycles, which is below the numerical precision threshold for meaningful updates.

In [ ]:
import math

FINETUNE_EPOCHS = 50
UNFREEZE_FROM_LAYER = 100

if not MODEL_FROZEN_PATH.exists():
    raise FileNotFoundError(f"Frozen model not found: {MODEL_FROZEN_PATH}")

# Load best frozen-phase model. No custom metric is needed because QWK is now a callback.
model = tf.keras.models.load_model(
    MODEL_FROZEN_PATH,
    compile=False
)

base_model = model.get_layer("efficientnetb0")
base_model.trainable = True

for layer in base_model.layers[:UNFREEZE_FROM_LAYER]:
    layer.trainable = False
for layer in base_model.layers[UNFREEZE_FROM_LAYER:]:
    layer.trainable = True

# Keep all BatchNorm layers frozen.
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

trainable_count = sum(1 for layer in base_model.layers if layer.trainable)
print(f"Trainable EfficientNetB0 layers: {trainable_count} / {len(base_model.layers)}")

steps_per_epoch_ft = math.ceil(len(X_train_fit) / BATCH_SIZE)

lr_schedule_finetune = tf.keras.optimizers.schedules.CosineDecayRestarts(
    initial_learning_rate=1e-4,
    first_decay_steps=steps_per_epoch_ft * 5,
    t_mul=2.0,
    m_mul=0.9,
    alpha=1e-3
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule_finetune),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks_finetune = [
    ValidationQWK(validation_data=(X_val, y_val), batch_size=BATCH_SIZE),

    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(MODEL_FINETUNE_PATH),
        monitor="val_qwk",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_qwk",
        mode="max",
        patience=15,
        restore_best_weights=True,
        verbose=1
    )
]

print("Starting fine-tuning...")

history_finetune = model.fit(
    X_train_fit,
    y_train_fit,
    validation_data=(X_val, y_val),
    epochs=FINETUNE_EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=TRAIN_CLASS_WEIGHT,
    callbacks=callbacks_finetune
)

history_finetune_dict = {
    k: [float(x) for x in v]
    for k, v in history_finetune.history.items()
}

with open(HISTORY_FINETUNE_PATH, "w") as f:
    json.dump(history_finetune_dict, f, indent=4)

print("Phase 2 complete. Best model saved to:", MODEL_FINETUNE_PATH)


## 10. Evaluation

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    cohen_kappa_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    auc
)
from sklearn.preprocessing import label_binarize

CLASS_NAMES = ["No DR", "Mild", "Moderate", "Severe", "Proliferative DR"]
CLASS_IDS = np.arange(NUM_CLASSES)

# Keep the test image ids so Grad-CAM and lesion-mask evaluation can be matched later.
test_id_codes = df.iloc[test_idx]["id_code"].astype(str).to_numpy()

# Prefer the fine-tuned model. If fine-tuning was skipped or failed, evaluate frozen model.
if MODEL_FINETUNE_PATH.exists():
    EVAL_MODEL_PATH = MODEL_FINETUNE_PATH
    print("Evaluating fine-tuned model on TEST set:", EVAL_MODEL_PATH)
elif MODEL_FROZEN_PATH.exists():
    EVAL_MODEL_PATH = MODEL_FROZEN_PATH
    print("Fine-tuned model not found. Evaluating frozen model on TEST set:", EVAL_MODEL_PATH)
else:
    raise FileNotFoundError("No saved model found for evaluation.")

best_model = tf.keras.models.load_model(
    EVAL_MODEL_PATH,
    compile=False
)

y_test_pred_probs = best_model.predict(
    X_test,
    batch_size=BATCH_SIZE,
    verbose=1
)

y_test_pred = np.argmax(y_test_pred_probs, axis=1)

# ---------- AUC ----------
# For multiclass grading, report one-vs-rest AUC per class plus macro/weighted OVR.
# OVO is also reported as a robustness check when sklearn supports it for the current labels.
y_test_bin = label_binarize(y_test, classes=CLASS_IDS)

auc_per_class_ovr = {}
for class_id, class_name in zip(CLASS_IDS, CLASS_NAMES):
    positives = int(y_test_bin[:, class_id].sum())
    negatives = int(len(y_test_bin) - positives)

    if positives > 0 and negatives > 0:
        auc_per_class_ovr[class_name] = float(
            roc_auc_score(y_test_bin[:, class_id], y_test_pred_probs[:, class_id])
        )
    else:
        auc_per_class_ovr[class_name] = None

def safe_auc_score(*args, **kwargs):
    try:
        return float(roc_auc_score(*args, **kwargs))
    except Exception as exc:
        print("AUC skipped:", exc)
        return None

auc_ovr_macro = safe_auc_score(
    y_test,
    y_test_pred_probs,
    labels=CLASS_IDS,
    multi_class="ovr",
    average="macro"
)

auc_ovr_weighted = safe_auc_score(
    y_test,
    y_test_pred_probs,
    labels=CLASS_IDS,
    multi_class="ovr",
    average="weighted"
)

auc_ovo_macro = safe_auc_score(
    y_test,
    y_test_pred_probs,
    labels=CLASS_IDS,
    multi_class="ovo",
    average="macro"
)

auc_ovo_weighted = safe_auc_score(
    y_test,
    y_test_pred_probs,
    labels=CLASS_IDS,
    multi_class="ovo",
    average="weighted"
)

auc_ovr_micro = safe_auc_score(
    y_test_bin,
    y_test_pred_probs,
    average="micro"
)

test_metrics = {
    "evaluated_model": str(EVAL_MODEL_PATH),
    "evaluation_set": "test",
    "prediction_type": "softmax probabilities converted to class labels by argmax",
    "accuracy": float(accuracy_score(y_test, y_test_pred)),
    "macro_f1": float(f1_score(y_test, y_test_pred, average="macro", zero_division=0)),
    "macro_precision": float(precision_score(y_test, y_test_pred, average="macro", zero_division=0)),
    "macro_recall": float(recall_score(y_test, y_test_pred, average="macro", zero_division=0)),
    "weighted_f1": float(f1_score(y_test, y_test_pred, average="weighted", zero_division=0)),
    "quadratic_weighted_kappa": float(cohen_kappa_score(y_test, y_test_pred, weights="quadratic")),
    "auc_ovr_macro": auc_ovr_macro,
    "auc_ovr_weighted": auc_ovr_weighted,
    "auc_ovr_micro": auc_ovr_micro,
    "auc_ovo_macro": auc_ovo_macro,
    "auc_ovo_weighted": auc_ovo_weighted,
    "auc_per_class_ovr": auc_per_class_ovr
}

with open(TEST_METRICS_PATH, "w") as f:
    json.dump(test_metrics, f, indent=4)

print("=" * 40)
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")
print("=" * 40)

# ---------- Classification report ----------
test_report = classification_report(
    y_test,
    y_test_pred,
    labels=CLASS_IDS,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

test_report_df = pd.DataFrame(test_report).transpose()
test_report_df.to_csv(TEST_REPORT_PATH)

# ---------- Confusion matrix ----------
test_cm = confusion_matrix(y_test, y_test_pred, labels=CLASS_IDS)
test_cm_norm = confusion_matrix(y_test, y_test_pred, labels=CLASS_IDS, normalize="true")

test_cm_df = pd.DataFrame(
    test_cm,
    index=[f"true_{c}" for c in CLASS_NAMES],
    columns=[f"pred_{c}" for c in CLASS_NAMES]
)

test_cm_norm_df = pd.DataFrame(
    test_cm_norm,
    index=[f"true_{c}" for c in CLASS_NAMES],
    columns=[f"pred_{c}" for c in CLASS_NAMES]
)

test_cm_df.to_csv(TEST_CM_PATH)
test_cm_norm_df.to_csv(TEST_CM_NORMALIZED_PATH)

fig, ax = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=test_cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, values_format="d", xticks_rotation=45, colorbar=False)
ax.set_title("Test Confusion Matrix — Counts")
plt.tight_layout()
plt.savefig(TEST_CM_RAW_PNG_PATH, dpi=200, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=test_cm_norm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, values_format=".2f", xticks_rotation=45, colorbar=True)
ax.set_title("Test Confusion Matrix — Row Normalized")
plt.tight_layout()
plt.savefig(TEST_CM_NORMALIZED_PNG_PATH, dpi=200, bbox_inches="tight")
plt.show()

# ---------- ROC curves ----------
plt.figure(figsize=(8, 7))

for class_id, class_name in zip(CLASS_IDS, CLASS_NAMES):
    positives = int(y_test_bin[:, class_id].sum())
    negatives = int(len(y_test_bin) - positives)

    if positives == 0 or negatives == 0:
        continue

    fpr, tpr, _ = roc_curve(y_test_bin[:, class_id], y_test_pred_probs[:, class_id])
    roc_auc_value = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{class_name} AUC={roc_auc_value:.3f}")

plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("One-vs-Rest ROC Curves on Test Set")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(TEST_ROC_CURVE_PATH, dpi=200, bbox_inches="tight")
plt.show()

print("Saved test metrics:", TEST_METRICS_PATH)
print("Saved test report:", TEST_REPORT_PATH)
print("Saved test confusion matrix:", TEST_CM_PATH)
print("Saved normalized test confusion matrix:", TEST_CM_NORMALIZED_PATH)
print("Saved raw confusion matrix plot:", TEST_CM_RAW_PNG_PATH)
print("Saved normalized confusion matrix plot:", TEST_CM_NORMALIZED_PNG_PATH)
print("Saved ROC curve plot:", TEST_ROC_CURVE_PATH)

print(test_report_df)
print(test_cm_df)
print(test_cm_norm_df)

## 11. Paper-Aligned XAI Evaluation

This section adds Grad-CAM heatmaps and optional quantitative localization checks.

Paper basis:
- Grad-CAM follows Selvaraju et al. (ICCV 2017), which uses gradients flowing into the final convolutional layer to produce class-discriminative localization maps.
- Pointing Game follows Zhang et al. (ECCV 2016): the explanation gets a hit when the most salient heatmap pixel falls inside a ground-truth annotation mask.
- IoU-style overlap is a stricter segmentation-style check: threshold the heatmap, compare it with the lesion mask, and compute intersection over union.
- DR lesion-mask evaluation requires pixel-level lesion masks. APTOS 2019 has severity labels only; datasets such as IDRiD provide masks for microaneurysms, hemorrhages, hard exudates, and soft exudates.

The Grad-CAM visualization cell works directly on APTOS. The Pointing Game and IoU cell will skip automatically unless lesion masks are placed in `LESION_MASK_DIR`.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from tensorflow.keras import layers

GRADCAM_DIR.mkdir(parents=True, exist_ok=True)

def find_last_conv_layer_name_for_gradcam(model, nested_model_name="efficientnetb0"):
    """Return the last 4D layer inside EfficientNetB0. Prefer top_conv if present."""
    base = model.get_layer(nested_model_name)

    try:
        base.get_layer("top_conv")
        return "top_conv"
    except ValueError:
        pass

    for layer in reversed(base.layers):
        try:
            if len(layer.output.shape) == 4:
                return layer.name
        except Exception:
            continue

    raise ValueError("Could not find a 4D convolutional layer for Grad-CAM.")


LAST_CONV_LAYER_NAME = find_last_conv_layer_name_for_gradcam(best_model)
print("Using Grad-CAM target layer:", LAST_CONV_LAYER_NAME)


def run_classification_head(model, base_outputs, training=False):
    """Run the part of the model after EfficientNetB0."""
    x = model.get_layer("gap")(base_outputs)
    x = model.get_layer("dense_256")(x)
    x = model.get_layer("bn_head")(x, training=training)
    x = model.get_layer("dropout_1")(x, training=training)
    x = model.get_layer("dense_128")(x)
    x = model.get_layer("dropout_2")(x, training=training)
    return model.get_layer("output")(x)


def make_gradcam_heatmap(
    model,
    image,
    class_index=None,
    last_conv_layer_name=LAST_CONV_LAYER_NAME
):
    """
    Generate a Grad-CAM heatmap for one image.

    image: one preprocessed image with shape (224, 224, 3), values in [0, 255].
    class_index:
      - None: explain the predicted class.
      - integer: explain a specific target class.
    """
    if image.ndim == 3:
        image_batch = np.expand_dims(image, axis=0)
    else:
        image_batch = image

    image_batch = tf.convert_to_tensor(image_batch, dtype=tf.float32)

    augmentation_layer = model.get_layer("data_augmentation")
    base_model = model.get_layer("efficientnetb0")
    target_layer = base_model.get_layer(last_conv_layer_name)

    conv_model = tf.keras.Model(
        inputs=base_model.input,
        outputs=[target_layer.output, base_model.output]
    )

    with tf.GradientTape() as tape:
        # data_augmentation is included in the trained model, but it is disabled with training=False.
        x = augmentation_layer(image_batch, training=False)
        conv_outputs, base_outputs = conv_model(x, training=False)
        tape.watch(conv_outputs)

        predictions = run_classification_head(model, base_outputs, training=False)

        if class_index is None:
            class_index = int(tf.argmax(predictions[0]).numpy())

        class_score = predictions[:, class_index]

    grads = tape.gradient(class_score, conv_outputs)

    if grads is None:
        raise RuntimeError("Gradients are None. Check that the target layer is connected to the classification output.")

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)

    heatmap = tf.maximum(heatmap, 0)
    max_value = tf.reduce_max(heatmap)

    if float(max_value.numpy()) > 0:
        heatmap = heatmap / max_value

    heatmap = heatmap.numpy().astype(np.float32)
    heatmap = cv2.resize(
        heatmap,
        (image_batch.shape[2], image_batch.shape[1]),
        interpolation=cv2.INTER_LINEAR
    )

    return heatmap, int(class_index), predictions.numpy()[0]


def overlay_gradcam(image, heatmap, alpha=0.35):
    """Overlay a normalized heatmap on a fundus image."""
    image_uint8 = np.clip(image, 0, 255).astype(np.uint8)
    heatmap_uint8 = np.uint8(255 * np.clip(heatmap, 0, 1))
    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(image_uint8, 1 - alpha, heatmap_color, alpha, 0)
    return overlay


def select_gradcam_indices(y_true, y_pred, per_class=2, max_examples=15, random_state=42):
    """
    Pick a small but useful set:
    - up to per_class correct examples for each class
    - up to per_class mistakes for each true class
    """
    rng = np.random.default_rng(random_state)
    selected = []

    for class_id in CLASS_IDS:
        correct = np.where((y_true == class_id) & (y_pred == class_id))[0]
        wrong = np.where((y_true == class_id) & (y_pred != class_id))[0]

        if len(correct) > 0:
            selected.extend(rng.choice(correct, size=min(per_class, len(correct)), replace=False).tolist())

        if len(wrong) > 0:
            selected.extend(rng.choice(wrong, size=min(per_class, len(wrong)), replace=False).tolist())

    selected = list(dict.fromkeys(selected))
    return selected[:max_examples]


GRADCAM_CLASS_MODE = "predicted"  # "predicted" explains the model decision; "true" explains the ground-truth class.
GRADCAM_EXAMPLES_PER_CLASS = 2
MAX_GRADCAM_EXAMPLES = 15

gradcam_indices = select_gradcam_indices(
    y_test,
    y_test_pred,
    per_class=GRADCAM_EXAMPLES_PER_CLASS,
    max_examples=MAX_GRADCAM_EXAMPLES
)

if len(gradcam_indices) == 0:
    print("No Grad-CAM examples selected.")
else:
    gradcam_records = []
    n = len(gradcam_indices)

    fig, axes = plt.subplots(n, 3, figsize=(12, 3.5 * n))

    if n == 1:
        axes = np.expand_dims(axes, axis=0)

    for row, local_idx in enumerate(gradcam_indices):
        image = X_test[local_idx]
        true_class = int(y_test[local_idx])
        pred_class = int(y_test_pred[local_idx])

        if GRADCAM_CLASS_MODE == "true":
            target_class = true_class
        else:
            target_class = pred_class

        heatmap, explained_class, probs = make_gradcam_heatmap(
            best_model,
            image,
            class_index=target_class
        )

        overlay = overlay_gradcam(image, heatmap)
        image_id = str(test_id_codes[local_idx])

        overlay_path = GRADCAM_DIR / (
            f"{image_id}_true-{true_class}_pred-{pred_class}_explained-{explained_class}_overlay.png"
        )
        heatmap_path = GRADCAM_DIR / (
            f"{image_id}_true-{true_class}_pred-{pred_class}_explained-{explained_class}_heatmap.png"
        )

        plt.imsave(overlay_path, overlay)
        plt.imsave(heatmap_path, heatmap, cmap="jet")

        axes[row, 0].imshow(np.clip(image / 255.0, 0.0, 1.0))
        axes[row, 0].set_title(f"{image_id}\ntrue={CLASS_NAMES[true_class]}")
        axes[row, 0].axis("off")

        axes[row, 1].imshow(heatmap, cmap="jet")
        axes[row, 1].set_title(f"Grad-CAM\nexplains={CLASS_NAMES[explained_class]}")
        axes[row, 1].axis("off")

        axes[row, 2].imshow(overlay)
        axes[row, 2].set_title(f"pred={CLASS_NAMES[pred_class]}\nconf={probs[pred_class]:.3f}")
        axes[row, 2].axis("off")

        gradcam_records.append({
            "test_local_index": int(local_idx),
            "id_code": image_id,
            "true_class_id": true_class,
            "true_class_name": CLASS_NAMES[true_class],
            "pred_class_id": pred_class,
            "pred_class_name": CLASS_NAMES[pred_class],
            "explained_class_id": explained_class,
            "explained_class_name": CLASS_NAMES[explained_class],
            "predicted_confidence": float(probs[pred_class]),
            "overlay_path": str(overlay_path),
            "heatmap_path": str(heatmap_path)
        })

    plt.tight_layout()

    gradcam_grid_path = GRADCAM_DIR / f"{EXPERIMENT_NAME}_gradcam_examples_grid.png"
    plt.savefig(gradcam_grid_path, dpi=200, bbox_inches="tight")
    plt.show()

    gradcam_index_path = GRADCAM_DIR / f"{EXPERIMENT_NAME}_gradcam_examples_index.csv"
    pd.DataFrame(gradcam_records).to_csv(gradcam_index_path, index=False)

    print("Saved Grad-CAM example grid:", gradcam_grid_path)
    print("Saved Grad-CAM example index:", gradcam_index_path)
    print("Saved individual Grad-CAM overlays to:", GRADCAM_DIR)

In [ ]:
import json
from tqdm.auto import tqdm

# Quantitative localization evaluation.
# This requires lesion masks. APTOS 2019 does not ship with lesion masks.
# Put masks in LESION_MASK_DIR, preferably with filenames containing the same id_code.
# The loader recursively combines all matching masks for an image into one binary lesion mask.

LESION_MASK_EXTENSIONS = [".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".npy"]
HEATMAP_TOP_PERCENT = 15.0  # threshold top 15% of heatmap pixels for IoU-style evaluation
MAX_XAI_EVAL_IMAGES = None  # set to an integer for a faster subset, e.g. 100


def get_crop_bbox_from_rgb(img_rgb, threshold=10):
    """Return crop box using the same black-border logic as preprocessing."""
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    valid = gray > threshold

    if not valid.any():
        h, w = gray.shape
        return 0, h, 0, w

    rows = np.where(valid.any(axis=1))[0]
    cols = np.where(valid.any(axis=0))[0]
    return int(rows[0]), int(rows[-1] + 1), int(cols[0]), int(cols[-1] + 1)


def resize_binary_mask_with_padding(mask, target_size=224):
    """Resize a binary mask with the same aspect-ratio padding used for images."""
    mask = (mask > 0).astype(np.uint8)

    h, w = mask.shape[:2]
    scale = target_size / max(h, w)

    new_w = int(round(w * scale))
    new_h = int(round(h * scale))

    resized = cv2.resize(
        mask,
        (new_w, new_h),
        interpolation=cv2.INTER_NEAREST
    )

    padded = np.zeros((target_size, target_size), dtype=np.uint8)

    y_off = (target_size - new_h) // 2
    x_off = (target_size - new_w) // 2

    padded[y_off:y_off + new_h, x_off:x_off + new_w] = resized

    return padded.astype(bool)


def read_mask_file(mask_path):
    """Read one mask file as a binary numpy array."""
    mask_path = Path(mask_path)

    if mask_path.suffix.lower() == ".npy":
        mask = np.load(mask_path)
    else:
        mask = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)

    if mask is None:
        return None

    if mask.ndim == 3:
        # If the mask is RGB/RGBA, combine non-zero channels.
        mask = np.max(mask, axis=-1)

    return (mask > 0).astype(np.uint8)


def preprocess_mask_to_model_space(mask, original_image_path, img_size=224):
    """
    Align lesion masks to the model input space.

    If mask has original image size, apply the same crop and padding as the fundus image.
    If mask already has model size, keep it.
    Otherwise, resize with nearest-neighbor as a fallback.
    """
    if mask.shape[:2] == (img_size, img_size):
        return (mask > 0).astype(bool)

    original_bgr = cv2.imread(str(original_image_path))

    if original_bgr is not None:
        original_rgb = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2RGB)

        if mask.shape[:2] == original_rgb.shape[:2]:
            y0, y1, x0, x1 = get_crop_bbox_from_rgb(original_rgb, threshold=10)
            mask = mask[y0:y1, x0:x1]
            return resize_binary_mask_with_padding(mask, target_size=img_size)

    # Fallback for already-cropped masks or unknown layouts.
    resized = cv2.resize(
        (mask > 0).astype(np.uint8),
        (img_size, img_size),
        interpolation=cv2.INTER_NEAREST
    )

    return resized.astype(bool)


def find_mask_files_for_id(id_code, mask_root):
    """Find all lesion mask files whose filename contains the test image id_code."""
    mask_root = Path(mask_root)

    if not mask_root.exists():
        return []

    id_code = str(id_code)
    matches = []

    for ext in LESION_MASK_EXTENSIONS:
        matches.extend(mask_root.rglob(f"{id_code}{ext}"))
        matches.extend(mask_root.rglob(f"{id_code}_*{ext}"))
        matches.extend(mask_root.rglob(f"*{id_code}*{ext}"))

    # De-duplicate while preserving order.
    return list(dict.fromkeys(matches))


def load_combined_lesion_mask(id_code, original_image_path, mask_root, img_size=224):
    """Load and OR-combine all matching lesion masks for one image."""
    mask_files = find_mask_files_for_id(id_code, mask_root)

    if len(mask_files) == 0:
        return None, []

    combined = np.zeros((img_size, img_size), dtype=bool)
    loaded_files = []

    for mask_file in mask_files:
        raw_mask = read_mask_file(mask_file)

        if raw_mask is None:
            continue

        processed_mask = preprocess_mask_to_model_space(
            raw_mask,
            original_image_path=original_image_path,
            img_size=img_size
        )

        combined |= processed_mask
        loaded_files.append(str(mask_file))

    if len(loaded_files) == 0 or combined.sum() == 0:
        return None, loaded_files

    return combined, loaded_files


def compute_pointing_game_and_iou(heatmap, lesion_mask, top_percent=15.0):
    """
    Compute paper-style localization metrics:
    - Pointing Game hit: argmax heatmap pixel is inside the lesion mask.
    - Energy inside mask: percentage of heatmap mass inside the lesion mask.
    - IoU/Dice/precision/recall after thresholding top heatmap pixels.
    """
    heatmap = np.asarray(heatmap, dtype=np.float32)
    lesion_mask = np.asarray(lesion_mask).astype(bool)

    if lesion_mask.sum() == 0:
        return None

    peak_y, peak_x = np.unravel_index(np.argmax(heatmap), heatmap.shape)
    pointing_hit = int(lesion_mask[peak_y, peak_x])

    heatmap_sum = float(heatmap.sum())
    if heatmap_sum > 0:
        energy_inside = float(heatmap[lesion_mask].sum() / heatmap_sum)
    else:
        energy_inside = 0.0

    if float(heatmap.max()) <= 0:
        heatmap_binary = np.zeros_like(lesion_mask, dtype=bool)
    else:
        threshold = np.percentile(heatmap, 100.0 - top_percent)
        heatmap_binary = heatmap >= threshold

    intersection = np.logical_and(heatmap_binary, lesion_mask).sum()
    union = np.logical_or(heatmap_binary, lesion_mask).sum()

    heatmap_area = heatmap_binary.sum()
    lesion_area = lesion_mask.sum()

    iou = float(intersection / union) if union > 0 else 0.0
    precision = float(intersection / heatmap_area) if heatmap_area > 0 else 0.0
    recall = float(intersection / lesion_area) if lesion_area > 0 else 0.0

    if precision + recall > 0:
        dice = float(2 * precision * recall / (precision + recall))
    else:
        dice = 0.0

    return {
        "pointing_hit": pointing_hit,
        "peak_y": int(peak_y),
        "peak_x": int(peak_x),
        "energy_inside_mask": energy_inside,
        "iou_top_percent": iou,
        "dice_top_percent": dice,
        "precision_top_percent": precision,
        "recall_top_percent": recall,
        "heatmap_top_percent": float(top_percent),
        "lesion_area_pixels": int(lesion_area),
        "heatmap_area_pixels": int(heatmap_area),
        "intersection_pixels": int(intersection),
        "union_pixels": int(union)
    }


if not LESION_MASK_DIR.exists():
    print("Skipping quantitative XAI evaluation.")
    print("LESION_MASK_DIR does not exist:", LESION_MASK_DIR)
    print("APTOS 2019 has severity labels but no lesion masks. Add IDRiD-style lesion masks or another pixel-annotated DR dataset to run Pointing Game and IoU.")
else:
    xai_rows = []
    missing_masks = 0

    eval_indices = np.arange(len(X_test))

    if MAX_XAI_EVAL_IMAGES is not None:
        eval_indices = eval_indices[:MAX_XAI_EVAL_IMAGES]

    for local_idx in tqdm(eval_indices, desc="Grad-CAM XAI mask evaluation"):
        image_id = str(test_id_codes[local_idx])
        original_image_path = train_images_dir / f"{image_id}.png"

        lesion_mask, mask_files = load_combined_lesion_mask(
            image_id,
            original_image_path=original_image_path,
            mask_root=LESION_MASK_DIR,
            img_size=IMG_SIZE
        )

        if lesion_mask is None:
            missing_masks += 1
            continue

        image = X_test[local_idx]
        true_class = int(y_test[local_idx])
        pred_class = int(y_test_pred[local_idx])

        # For explainability evaluation, explain the model's predicted class.
        heatmap, explained_class, probs = make_gradcam_heatmap(
            best_model,
            image,
            class_index=pred_class
        )

        metrics = compute_pointing_game_and_iou(
            heatmap,
            lesion_mask,
            top_percent=HEATMAP_TOP_PERCENT
        )

        if metrics is None:
            continue

        row = {
            "test_local_index": int(local_idx),
            "id_code": image_id,
            "true_class_id": true_class,
            "true_class_name": CLASS_NAMES[true_class],
            "pred_class_id": pred_class,
            "pred_class_name": CLASS_NAMES[pred_class],
            "explained_class_id": int(explained_class),
            "explained_class_name": CLASS_NAMES[int(explained_class)],
            "predicted_confidence": float(probs[pred_class]),
            "mask_file_count": int(len(mask_files)),
            "mask_files": "|".join(mask_files)
        }

        row.update(metrics)
        xai_rows.append(row)

    if len(xai_rows) == 0:
        print("No images with usable lesion masks were found.")
        print("Missing masks for images checked:", missing_masks)
    else:
        xai_df = pd.DataFrame(xai_rows)
        xai_df.to_csv(XAI_METRICS_PATH, index=False)

        xai_summary = {
            "evaluated_model": str(EVAL_MODEL_PATH),
            "lesion_mask_dir": str(LESION_MASK_DIR),
            "n_test_images_checked": int(len(eval_indices)),
            "n_images_with_usable_masks": int(len(xai_df)),
            "n_missing_or_empty_masks": int(missing_masks),
            "pointing_game_accuracy": float(xai_df["pointing_hit"].mean()),
            "mean_energy_inside_mask": float(xai_df["energy_inside_mask"].mean()),
            f"mean_iou_top_{HEATMAP_TOP_PERCENT:g}_percent": float(xai_df["iou_top_percent"].mean()),
            f"mean_dice_top_{HEATMAP_TOP_PERCENT:g}_percent": float(xai_df["dice_top_percent"].mean()),
            f"mean_precision_top_{HEATMAP_TOP_PERCENT:g}_percent": float(xai_df["precision_top_percent"].mean()),
            f"mean_recall_top_{HEATMAP_TOP_PERCENT:g}_percent": float(xai_df["recall_top_percent"].mean())
        }

        with open(XAI_SUMMARY_PATH, "w") as f:
            json.dump(xai_summary, f, indent=4)

        print("Saved Grad-CAM XAI metrics:", XAI_METRICS_PATH)
        print("Saved Grad-CAM XAI summary:", XAI_SUMMARY_PATH)
        print(json.dumps(xai_summary, indent=4))

        display(xai_df.head())

## 12. Training Curves

In [ ]:
import json
import matplotlib.pyplot as plt


def plot_history(history, title):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(title)

    if "accuracy" in history:
        axes[0].plot(history["accuracy"], label="train")
    if "val_accuracy" in history:
        axes[0].plot(history["val_accuracy"], label="val")
    axes[0].set_title("Accuracy")
    axes[0].legend()

    if "loss" in history:
        axes[1].plot(history["loss"], label="train")
    if "val_loss" in history:
        axes[1].plot(history["val_loss"], label="val")
    axes[1].set_title("Loss")
    axes[1].legend()

    # QWK is produced by the callback as validation-only metric.
    if "val_qwk" in history:
        axes[2].plot(history["val_qwk"], label="val_qwk")
    axes[2].set_title("Quadratic Weighted Kappa")
    axes[2].legend()

    plt.tight_layout()
    plt.show()

if HISTORY_FROZEN_PATH.exists():
    with open(HISTORY_FROZEN_PATH) as f:
        h_frozen = json.load(f)
    plot_history(h_frozen, "Phase 1 — Frozen Backbone")
else:
    print("Frozen history not found:", HISTORY_FROZEN_PATH)

if HISTORY_FINETUNE_PATH.exists():
    with open(HISTORY_FINETUNE_PATH) as f:
        h_finetune = json.load(f)
    plot_history(h_finetune, "Phase 2 — Fine-Tuning")
else:
    print("Fine-tune history not found:", HISTORY_FINETUNE_PATH)


## 13. Experiment Config (Documentation)

In [ ]:
# Convert class_weights to plain JSON-safe values.
class_weights_json = {
    str(k): float(v)
    for k, v in class_weights.items()
}

smote_distribution_before = {
    str(k): int(v)
    for k, v in Counter(y_train).items()
}

smote_distribution_after = {
    str(k): int(v)
    for k, v in Counter(y_train_fit).items()
}

smote_sampling_strategy_json = {
    str(k): int(v)
    for k, v in sampling_strategy.items()
}

experiment_config = {
    "experiment_name": EXPERIMENT_NAME,
    "goal": "Evaluate conservative SMOTE for APTOS 2019 DR severity classification using RGB + LAB-CLAHE and EfficientNetB0, then add paper-aligned evaluation and XAI localization checks.",
    "fixes_applied": {
        "fix_1_preprocessing": "RGB + LAB-CLAHE. Preserves color information while enhancing luminance contrast.",
        "fix_2_split_before_smote": "Train/validation/test split is created before SMOTE to avoid data leakage.",
        "fix_3_smote_train_only": "SMOTE is applied only to the training split. Validation and test remain untouched.",
        "fix_4_conservative_strategy": "Classes 1, 3, and 4 are oversampled only up to the class-2 Moderate count, not to the No-DR majority count.",
        "fix_5_smote_k_neighbors": f"SMOTE uses k_neighbors={k_neighbors}, capped by the smallest targeted class size.",
        "fix_6_no_class_weight_with_smote": "class_weight is disabled during SMOTE training to avoid double compensation.",
        "fix_7_lr_schedule": "CosineDecay for frozen phase and CosineDecayRestarts for fine-tuning.",
        "fix_8_qwk_callback": "QWK is computed with a validation callback instead of a Keras Metric.",
        "fix_9_test_evaluation": "Final metrics are calculated on a held-out test set.",
        "fix_10_auc": "Evaluation reports multiclass ROC-AUC: one-vs-rest per class plus OVR/OVO macro and weighted scores.",
        "fix_11_confusion_matrix_plots": "Evaluation saves raw and row-normalized confusion-matrix plots.",
        "fix_12_gradcam_xai": "Grad-CAM heatmaps are generated from the last EfficientNetB0 convolutional layer for selected test images.",
        "fix_13_pointing_game_iou": "Optional quantitative explanation evaluation computes Pointing Game, energy inside mask, IoU, Dice, precision, and recall when lesion masks are available."
    },
    "dataset": "APTOS 2019",
    "important_limitation": "APTOS 2019 has image-level severity labels only. Quantitative lesion-localization evaluation requires external pixel-level lesion masks such as IDRiD.",
    "image_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "original_class_weights_not_used_with_smote": class_weights_json,
    "smote_distribution_before": smote_distribution_before,
    "smote_distribution_after": smote_distribution_after,
    "smote_sampling_strategy": smote_sampling_strategy_json,
    "smote_k_neighbors": int(k_neighbors),
    "train_class_weight": TRAIN_CLASS_WEIGHT,
    "unfreeze_from_layer": UNFREEZE_FROM_LAYER,
    "frozen_epochs": FROZEN_EPOCHS,
    "finetune_epochs": FINETUNE_EPOCHS,
    "evaluation_outputs": {
        "test_metrics_json": str(TEST_METRICS_PATH),
        "classification_report_csv": str(TEST_REPORT_PATH),
        "confusion_matrix_csv": str(TEST_CM_PATH),
        "confusion_matrix_normalized_csv": str(TEST_CM_NORMALIZED_PATH),
        "confusion_matrix_raw_png": str(TEST_CM_RAW_PNG_PATH),
        "confusion_matrix_normalized_png": str(TEST_CM_NORMALIZED_PNG_PATH),
        "roc_curve_png": str(TEST_ROC_CURVE_PATH),
        "gradcam_dir": str(GRADCAM_DIR),
        "xai_metrics_csv": str(XAI_METRICS_PATH),
        "xai_summary_json": str(XAI_SUMMARY_PATH),
        "lesion_mask_dir": str(LESION_MASK_DIR)
    },
    "paper_basis": [
        {
            "name": "Selvaraju et al. (2017), Grad-CAM: Visual Explanations from Deep Networks via Gradient-Based Localization.",
            "used_for": "Grad-CAM heatmap generation from the final convolutional layer."
        },
        {
            "name": "Zhang et al. (2016), Top-down Neural Attention by Excitation Backprop.",
            "used_for": "Pointing Game localization metric."
        },
        {
            "name": "Porwal et al. (2018), Indian Diabetic Retinopathy Image Dataset (IDRiD).",
            "used_for": "Lesion-mask based evaluation: microaneurysms, hemorrhages, hard exudates, and soft exudates."
        },
        {
            "name": "Dharrao et al. (2025), AI-driven detection and classification of diabetic retinopathy stages using EfficientNetB0.",
            "used_for": "EfficientNetB0 DR severity-classification context."
        },
        {
            "name": "Recent DR XAI reviews/studies.",
            "used_for": "Justification for moving beyond visual-only heatmap inspection toward quantitative explanation evaluation."
        }
    ]
}

with open(CONFIG_PATH, "w") as f:
    json.dump(experiment_config, f, indent=4)

print("Config saved to:", CONFIG_PATH)
print(json.dumps(experiment_config, indent=4))